# Colab Session: thx02-bench
Generated from colab-cli history log.

**Session Created**: 2026-11-06 16:01:25
- Endpoint: `gpu-t4-s-usw4b1-1vuep64surjrw`
- Hardware: `T4`

### Automation: install (2026-11-06 16:01:56)

In [ ]:
import subprocess
import sys


def install():
    packages = ["thx02[local]"]
    try:
        subprocess.check_call(["uv", "pip", "install", "--system"] + packages)
        print("Installation Complete (via uv)!")
    except:
        subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)
        print("Installation Complete (via pip)!")


install()

In [ ]:
# Result of previous automation

Installation Complete (via uv)!


*File Operation*: `upload` on `/content/thx02_spike.jsonl`

*File Operation*: `upload` on `/content/colab_thx02_spike.py`

In [ ]:
#!/usr/bin/env python3
"""Run THX-02 benchmark on Google Colab with GPU (T4/L4/A100)."""

import json
import os
import time


def main():
    print("=" * 60)
    print("THX-02 Colab GPU Benchmark")
    print("=" * 60)

    # 1. Environment & CUDA Inspection
    import torch

    cuda_avail = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_avail else "No GPU (CPU)"
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if cuda_avail else 0.0
    print(f"CUDA Available: {cuda_avail}")
    print(f"Device: {device_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"Python Version: {sys.version}")

    # 2. Check / Install dependencies
    try:
        import thx02

        print(f"thx02 version: {thx02.__version__}")
    except ImportError:
        print("Installing thx02[local]...")
        import subprocess

        subprocess.check_call([sys.executable, "-m", "pip", "install", "thx02[local]"])
        import thx02

    from thx02 import THX02

    # 3. Model Loading
    print("\n--- Loading THX-02 on GPU ---")
    t0_load = time.perf_counter()
    device = "cuda" if cuda_avail else "cpu"
    model = THX02.load("HAL-X-AI/THX-02", device=device)
    t_load = time.perf_counter() - t0_load
    print(f"Model loaded in {t_load:.2f}s on {device}")

    if cuda_avail:
        mem_allocated = torch.cuda.memory_allocated(0) / (1024**3)
        mem_reserved = torch.cuda.memory_reserved(0) / (1024**3)
        print(f"GPU VRAM Allocated: {mem_allocated:.2f} GB, Reserved: {mem_reserved:.2f} GB")

    # 4. Single Test Case (Warmup + Validation)
    print("\n--- Test 1: Single Canonical Decision ---")
    sample_text = (
        "SENTENÇA. O autor pediu a restituição de valores. "
        "Ante o exposto, JULGO PROCEDENTE o pedido inicial, integralmente, "
        "com resolução do mérito."
    )
    t0_inf = time.perf_counter()
    pred_outcome = model.predict(
        sample_text,
        question="Qual é o resultado: procedente, improcedente, parcialmente procedente ou desconhecido?",
        choices=["procedente", "improcedente", "parcialmente procedente", "desconhecido"],
    )
    t_inf_warmup = time.perf_counter() - t0_inf
    print(f"Warmup Latency: {t_inf_warmup:.3f}s")
    print(f"Outcome Choice: {pred_outcome.choice}")
    print(f"Outcome Probabilities: {pred_outcome.probabilities}")

    # 5. Benchmark 12 Spike Fixture Decisions
    print("\n--- Benchmark: Fixture Decisions (tests/fixtures/thx02_spike.jsonl) ---")
    fixture_path = "tests/fixtures/thx02_spike.jsonl"
    if not os.path.exists(fixture_path):
        fixture_path = "/content/thx02_spike.jsonl"

    records = []
    if os.path.exists(fixture_path):
        with open(fixture_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        print(f"Loaded {len(records)} test cases from {fixture_path}")
    else:
        print("Warning: Fixture file not found, creating synthetic sample cases.")
        records = [
            {
                "id": "s01",
                "text": sample_text,
                "expected": {"outcome": "procedente", "winner": "autor"},
            }
        ]

    results = []
    latencies = []
    correct_outcome = 0
    correct_winner = 0

    outcome_choices = ["procedente", "improcedente", "parcialmente procedente", "desconhecido"]
    winner_choices = ["autor", "réu", "ambos", "nenhum"]

    for r in records:
        rec_id = r.get("id", "unknown")
        text = r.get("text", "")
        expected = r.get("expected", {})

        t0 = time.perf_counter()
        out_res = model.predict(
            text,
            question="Qual é o resultado: procedente, improcedente, parcialmente procedente ou desconhecido?",
            choices=outcome_choices,
        )
        win_res = model.predict(
            text, question="Qual polo venceu a demanda judicial?", choices=winner_choices
        )
        elapsed = time.perf_counter() - t0
        latencies.append(elapsed)

        is_out_correct = out_res.choice == expected.get("outcome")
        is_win_correct = win_res.choice == expected.get("winner")
        if is_out_correct:
            correct_outcome += 1
        if is_win_correct:
            correct_winner += 1

        print(
            f"[{rec_id}] Latency: {elapsed:.3f}s | Outcome: {out_res.choice} (exp: {expected.get('outcome')}) | Winner: {win_res.choice} (exp: {expected.get('winner')})"
        )

        results.append(
            {
                "id": rec_id,
                "latency_seconds": elapsed,
                "outcome_pred": out_res.choice,
                "outcome_probs": out_res.probabilities,
                "outcome_expected": expected.get("outcome"),
                "outcome_correct": is_out_correct,
                "winner_pred": win_res.choice,
                "winner_probs": win_res.probabilities,
                "winner_expected": expected.get("winner"),
                "winner_correct": is_win_correct,
            }
        )

    # Summary Statistics
    latencies.sort()
    n = len(latencies)
    p50 = latencies[int(n * 0.50)]
    p95 = latencies[min(int(n * 0.95), n - 1)]
    p99 = latencies[min(int(n * 0.99), n - 1)]
    avg_lat = sum(latencies) / n

    summary = {
        "device": device_name,
        "cuda": cuda_avail,
        "vram_gb": vram_gb,
        "total_cases": n,
        "outcome_accuracy": correct_outcome / n if n else 0.0,
        "winner_accuracy": correct_winner / n if n else 0.0,
        "latency_p50_s": p50,
        "latency_p95_s": p95,
        "latency_p99_s": p99,
        "latency_avg_s": avg_lat,
        "decisions_per_sec": 1.0 / avg_lat if avg_lat > 0 else 0.0,
        "model_load_time_s": t_load,
    }

    print("\n" + "=" * 60)
    print("BENCHMARK SUMMARY")
    print("=" * 60)
    for k, v in summary.items():
        print(f"{k}: {v}")

    # Save output files
    os.makedirs("/content/output", exist_ok=True)
    with open("/content/output/colab_thx02_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    with open("/content/output/colab_thx02_results.jsonl", "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("\nResults successfully saved to /content/output/")


if __name__ == "__main__":
    main()

THX-02 Colab GPU Benchmark


CUDA Available: True
Device: Tesla T4
Total VRAM: 14.56 GB
PyTorch Version: 2.11.0+cu130
Python Version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
thx02 version: 1.0.0

--- Loading THX-02 on GPU ---


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Model loaded in 350.41s on cuda
GPU VRAM Allocated: 8.55 GB, Reserved: 8.59 GB

--- Test 1: Single Canonical Decision ---


AttributeError: 'THX02' object has no attribute 'predict'

In [ ]:
#!/usr/bin/env python3
"""Run THX-02 benchmark on Google Colab with GPU (Tesla T4).

Reuses the exact canonical questions and decoding logic from PR #1687.
"""

from __future__ import annotations


# Reusable question dictionary matching spike_thx02.py exactly
QUESTIONS = {
    "decision_type": {
        "type": "choice",
        "instructions": (
            "Classifique o tipo do ato JUDICIAL efetivamente proferido neste documento, "
            "não as decisões citadas em relatório ou fundamentação."
        ),
        "criteria": {
            "sentenca": "Sentença: juiz decide definitivamente um pedido ou extingue o processo no grau.",
            "acordao": "Acórdão: julgamento colegiado de recurso ou ação originária.",
            "interlocutoria": "Decisão interlocutória: resolve questão incidental, não encerra o mérito da causa.",
            "despacho": "Despacho de mero expediente, sem julgamento de pedido ou recurso.",
            "outro": "Ato diferente, peça incompleta ou documento sem decisão identificável.",
        },
    },
    "outcome": {
        "type": "choice",
        "instructions": (
            "Determine exclusivamente o resultado OPERATIVO deste ato, "
            "não pedidos das partes nem decisões anteriores. "
            "Na sentença, classifique o pedido; no acórdão recursal, classifique o RECURSO. "
            "Se não houver dispositivo suficiente, use unknown."
        ),
        "criteria": {
            "procedente": "Pedido inicial julgado integralmente procedente no dispositivo.",
            "improcedente": "Pedido inicial julgado improcedente no dispositivo.",
            "parcialmente_procedente": "Pedido inicial julgado parcialmente procedente no dispositivo.",
            "acordo": "Acordo ou transação homologados judicialmente.",
            "extinto_sem_merito": "Extinção do processo sem resolução do mérito, por motivo processual.",
            "provido": "Recurso conhecido e integralmente provido no dispositivo.",
            "nao_provido": "Recurso conhecido e não provido (desprovido) no dispositivo.",
            "parcialmente_provido": "Recurso conhecido e parcialmente provido no dispositivo.",
            "nao_conhecido": "Recurso não conhecido (inadmissibilidade).",
            "prejudicado": "Recurso julgado prejudicado ou perda superveniente de objeto.",
            "unknown": "Não há julgamento conclusivo do pedido/recurso, ou é impossível determinar o resultado.",
        },
    },
    "recorrente_polo": {
        "type": "choice",
        "instructions": (
            "Qual polo interpôs o recurso OBJETO do dispositivo? "
            "Autor/requerente do pedido inicial = A; réu/requerido = P. "
            "Não confunda recorrente com parte vencedora."
        ),
        "criteria": {
            "A": "Somente o autor/parte ativa interpôs o recurso analisado.",
            "P": "Somente o réu/parte passiva interpôs o recurso analisado.",
            "ambos": "Ambos os polos interpuseram recursos analisados em conjunto.",
            "nao_aplicavel": "Não há recurso, trata-se de sentença ou outro ato não recursal.",
            "unknown": "Não é possível identificar com segurança o polo recorrente.",
        },
    },
    "winner_direct": {
        "type": "choice",
        "instructions": (
            "Considerando pedido inicial, polo recorrente e dispositivo atual, "
            "qual polo venceu materialmente a controvérsia? "
            "Sem decisão de mérito, vitória parcial ambígua ou dados insuficientes: unknown."
        ),
        "criteria": {
            "A": "Polo ativo/autor venceu integralmente o pedido ou preservou essa vitória em recurso.",
            "P": "Polo passivo/réu venceu integralmente o pedido ou preservou essa vitória em recurso.",
            "draw": "Acordo homologado, sem polo vencedor unilateral.",
            "unknown": "Sem vencedor material unívoco, decisão incidental, resultado parcial ou texto insuficiente.",
        },
    },
}

OUTCOME_MAP = {
    "procedente": "procedente",
    "improcedente": "improcedente",
    "parcialmente_procedente": "parcialmente procedente",
    "acordo": "acordo",
    "extinto_sem_merito": "extinto sem mérito",
    "provido": "provido",
    "nao_provido": "não provido",
    "parcialmente_provido": "parcialmente provido",
    "nao_conhecido": "não conhecido",
    "prejudicado": "prejudicado",
    "unknown": "unknown",
}


def resolve_winner(outcome: str, recorrente_polo: str | None) -> str:
    if outcome == "procedente":
        return "A"
    if outcome == "improcedente":
        return "P"
    if outcome == "acordo":
        return "draw"
    if outcome in {"provido", "não provido"} and recorrente_polo in {"A", "P"}:
        if outcome == "provido":
            return recorrente_polo
        return "P" if recorrente_polo == "A" else "A"
    return "unknown"


def main():
    print("=" * 65)
    print("THX-02 COLAB GPU BENCHMARK (NVIDIA TESLA T4)")
    print("=" * 65)

    import torch

    cuda_avail = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_avail else "No GPU"
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if cuda_avail else 0.0

    print(f"CUDA Available: {cuda_avail}")
    print(f"Device: {device_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"Python Version: {sys.version.split()[0]}")

    from thx02 import THX02

    # 1. Carregamento do modelo (usando pesos em cache na VM)
    print("\n--- [1] Carregando modelo THX-02 em GPU ---")
    t0_load = time.perf_counter()
    model = THX02.load("HAL-X-AI/THX-02", device="cuda" if cuda_avail else "cpu")
    t_load = time.perf_counter() - t0_load
    print(f"Modelo carregado com sucesso em {t_load:.2f}s!")

    if cuda_avail:
        mem_alloc = torch.cuda.memory_allocated(0) / (1024**3)
        mem_res = torch.cuda.memory_reserved(0) / (1024**3)
        print(f"VRAM alocada: {mem_alloc:.2f} GB | VRAM reservada: {mem_res:.2f} GB")

    # 2. Teste 1: Sentença canônica mínima (Aquecimento da GPU)
    print("\n--- [2] Teste Canônico Mínimo (Warmup) ---")
    sample_text = (
        "SENTENÇA. O autor pediu a restituição de valores. "
        "Ante o exposto, JULGO PROCEDENTE o pedido inicial, integralmente, "
        "com resolução do mérito."
    )
    t0_inf = time.perf_counter()
    warmup_res = model.decide(state=sample_text, questions=QUESTIONS, think=False, tta=1)
    t_warmup = time.perf_counter() - t0_inf
    print(f"Latência do aquecimento: {t_warmup:.3f}s ({t_warmup * 1000:.1f} ms)!")
    print(f"Resposta do aquecimento:\n{json.dumps(warmup_res, indent=2, ensure_ascii=False)}")

    # 3. Teste Fixture (12 Casos do CausaGanha)
    print("\n--- [3] Benchmark Completo da Fixture (12 casos) ---")
    fixture_path = "/content/thx02_spike.jsonl"
    if not os.path.exists(fixture_path):
        fixture_path = "tests/fixtures/thx02_spike.jsonl"

    documents = []
    with open(fixture_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                documents.append(json.loads(line))
    print(f"Carregados {len(documents)} documentos da fixture {fixture_path}.")

    results = []
    latencies = []
    correct_outcome = 0
    correct_winner = 0

    for i, doc in enumerate(documents, start=1):
        doc_id = doc["id"]
        text = doc["text"]
        gold = doc.get("gold", {})
        gold_out = gold.get("outcome")
        gold_win = gold.get("winner")

        t0 = time.perf_counter()
        raw_res = model.decide(state=text, questions=QUESTIONS, think=False, tta=1)
        elapsed_s = time.perf_counter() - t0
        elapsed_ms = elapsed_s * 1000.0
        latencies.append(elapsed_ms)

        # Decodificação
        answers = raw_res.get("answers", {})
        raw_out = answers.get("outcome", {}).get("choice", "unknown")
        outcome = OUTCOME_MAP.get(raw_out, "unknown")
        prob_out = answers.get("outcome", {}).get("probability", 0.0)

        raw_rec = answers.get("recorrente_polo", {}).get("choice")
        winner_direct = answers.get("winner_direct", {}).get("choice")
        resolved_win = resolve_winner(outcome, raw_rec)
        winner = winner_direct if winner_direct in {"A", "P", "draw"} else resolved_win

        match_out = outcome == gold_out
        match_win = winner == gold_win
        if match_out:
            correct_outcome += 1
        if match_win:
            correct_winner += 1

        mark_out = "✓" if match_out else "✗"
        mark_win = "✓" if match_win else "✗"
        print(
            f"[{i:02d}/{len(documents):02d}] {doc_id} ({len(text)} chars) -> Latência: {elapsed_ms:.1f} ms | Resultado: {outcome} ({prob_out:.1%}) [{mark_out}] | Vencedor: {winner} [{mark_win}]"
        )

        results.append(
            {
                "id": doc_id,
                "latency_ms": elapsed_ms,
                "outcome": outcome,
                "outcome_probability": prob_out,
                "gold_outcome": gold_out,
                "match_outcome": match_out,
                "winner": winner,
                "gold_winner": gold_win,
                "match_winner": match_win,
                "raw_response": raw_res,
            }
        )

    # Estatísticas
    latencies.sort()
    n = len(latencies)
    p50 = latencies[int(n * 0.50)]
    p95 = latencies[min(int(n * 0.95), n - 1)]
    p99 = latencies[min(int(n * 0.99), n - 1)]
    avg_ms = sum(latencies) / n

    summary = {
        "hardware": device_name,
        "cuda_available": cuda_avail,
        "vram_gb": vram_gb,
        "total_documents": n,
        "outcome_accuracy": correct_outcome / n if n else 0.0,
        "winner_accuracy": correct_winner / n if n else 0.0,
        "latency_p50_ms": p50,
        "latency_p95_ms": p95,
        "latency_p99_ms": p99,
        "latency_avg_ms": avg_ms,
        "throughput_decisions_per_sec": 1000.0 / avg_ms if avg_ms > 0 else 0.0,
        "model_load_time_seconds": t_load,
    }

    print("\n" + "=" * 65)
    print("RESUMO DO BENCHMARK THX-02 NA GPU (COLAB T4)")
    print("=" * 65)
    for k, v in summary.items():
        print(f"{k}: {v}")

    # Salva arquivos de saída
    os.makedirs("/content/output", exist_ok=True)
    with open("/content/output/colab_thx02_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    with open("/content/output/colab_thx02_results.jsonl", "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("\nResultados gravados em /content/output/colab_thx02_summary.json e results.jsonl!")


if __name__ == "__main__":
    main()

THX-02 COLAB GPU BENCHMARK (NVIDIA TESLA T4)
CUDA Available: True
Device: Tesla T4
Total VRAM: 14.56 GB
PyTorch Version: 2.11.0+cu130
Python Version: 3.13.16

--- [1] Carregando modelo THX-02 em GPU ---


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 40.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 39.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 14.37 GiB is allocated by PyTorch, and 54.04 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
#!/usr/bin/env python3
"""Run THX-02 benchmark on Google Colab with GPU (Tesla T4).

Reuses the exact canonical questions and decoding logic from PR #1687.
Optimized for 14.5 GB T4 VRAM with explicit bfloat16 and memory management.
"""

from __future__ import annotations


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Reusable question dictionary matching spike_thx02.py exactly
QUESTIONS = {
    "decision_type": {
        "type": "choice",
        "instructions": (
            "Classifique o tipo do ato JUDICIAL efetivamente proferido neste documento, "
            "não as decisões citadas em relatório ou fundamentação."
        ),
        "criteria": {
            "sentenca": "Sentença: juiz decide definitivamente um pedido ou extingue o processo no grau.",
            "acordao": "Acórdão: julgamento colegiado de recurso ou ação originária.",
            "interlocutoria": "Decisão interlocutória: resolve questão incidental, não encerra o mérito da causa.",
            "despacho": "Despacho de mero expediente, sem julgamento de pedido ou recurso.",
            "outro": "Ato diferente, peça incompleta ou documento sem decisão identificável.",
        },
    },
    "outcome": {
        "type": "choice",
        "instructions": (
            "Determine exclusivamente o resultado OPERATIVO deste ato, "
            "não pedidos das partes nem decisões anteriores. "
            "Na sentença, classifique o pedido; no acórdão recursal, classifique o RECURSO. "
            "Se não houver dispositivo suficiente, use unknown."
        ),
        "criteria": {
            "procedente": "Pedido inicial julgado integralmente procedente no dispositivo.",
            "improcedente": "Pedido inicial julgado improcedente no dispositivo.",
            "parcialmente_procedente": "Pedido inicial julgado parcialmente procedente no dispositivo.",
            "acordo": "Acordo ou transação homologados judicialmente.",
            "extinto_sem_merito": "Extinção do processo sem resolução do mérito, por motivo processual.",
            "provido": "Recurso conhecido e integralmente provido no dispositivo.",
            "nao_provido": "Recurso conhecido e não provido (desprovido) no dispositivo.",
            "parcialmente_provido": "Recurso conhecido e parcialmente provido no dispositivo.",
            "nao_conhecido": "Recurso não conhecido (inadmissibilidade).",
            "prejudicado": "Recurso julgado prejudicado ou perda superveniente de objeto.",
            "unknown": "Não há julgamento conclusivo do pedido/recurso, ou é impossível determinar o resultado.",
        },
    },
    "recorrente_polo": {
        "type": "choice",
        "instructions": (
            "Qual polo interpôs o recurso OBJETO do dispositivo? "
            "Autor/requerente do pedido inicial = A; réu/requerido = P. "
            "Não confunda recorrente com parte vencedora."
        ),
        "criteria": {
            "A": "Somente o autor/parte ativa interpôs o recurso analisado.",
            "P": "Somente o réu/parte passiva interpôs o recurso analisado.",
            "ambos": "Ambos os polos interpuseram recursos analisados em conjunto.",
            "nao_aplicavel": "Não há recurso, trata-se de sentença ou outro ato não recursal.",
            "unknown": "Não é possível identificar com segurança o polo recorrente.",
        },
    },
    "winner_direct": {
        "type": "choice",
        "instructions": (
            "Considerando pedido inicial, polo recorrente e dispositivo atual, "
            "qual polo venceu materialmente a controvérsia? "
            "Sem decisão de mérito, vitória parcial ambígua ou dados insuficientes: unknown."
        ),
        "criteria": {
            "A": "Polo ativo/autor venceu integralmente o pedido ou preservou essa vitória em recurso.",
            "P": "Polo passivo/réu venceu integralmente o pedido ou preservou essa vitória em recurso.",
            "draw": "Acordo homologado, sem polo vencedor unilateral.",
            "unknown": "Sem vencedor material unívoco, decisão incidental, resultado parcial ou texto insuficiente.",
        },
    },
}

OUTCOME_MAP = {
    "procedente": "procedente",
    "improcedente": "improcedente",
    "parcialmente_procedente": "parcialmente procedente",
    "acordo": "acordo",
    "extinto_sem_merito": "extinto sem mérito",
    "provido": "provido",
    "nao_provido": "não provido",
    "parcialmente_provido": "parcialmente provido",
    "nao_conhecido": "não conhecido",
    "prejudicado": "prejudicado",
    "unknown": "unknown",
}


def resolve_winner(outcome: str, recorrente_polo: str | None) -> str:
    if outcome == "procedente":
        return "A"
    if outcome == "improcedente":
        return "P"
    if outcome == "acordo":
        return "draw"
    if outcome in {"provido", "não provido"} and recorrente_polo in {"A", "P"}:
        if outcome == "provido":
            return recorrente_polo
        return "P" if recorrente_polo == "A" else "A"
    return "unknown"


def main():
    print("=" * 65)
    print("THX-02 COLAB GPU BENCHMARK (NVIDIA TESLA T4)")
    print("=" * 65)

    import torch

    cuda_avail = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_avail else "No GPU"
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if cuda_avail else 0.0

    print(f"CUDA Available: {cuda_avail}")
    print(f"Device: {device_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"Python Version: {sys.version.split()[0]}")

    # Força default dtype em bfloat16 para evitar OOM de float32
    torch.set_default_dtype(torch.bfloat16)

    # Monkeypatch Qwen3_5Model para garantir torch_dtype=bfloat16 e device_map
    from transformers import Qwen3_5Model

    orig_from_pretrained = Qwen3_5Model.from_pretrained

    @classmethod
    def patched_from_pretrained(cls, *args, **kwargs):
        kwargs["torch_dtype"] = torch.bfloat16
        return orig_from_pretrained(*args, **kwargs)

    Qwen3_5Model.from_pretrained = patched_from_pretrained

    from thx02 import THX02

    # 1. Carregamento do modelo
    print("\n--- [1] Carregando modelo THX-02 em GPU (bfloat16) ---")
    t0_load = time.perf_counter()
    model = THX02.load("HAL-X-AI/THX-02", device="cuda" if cuda_avail else "cpu")
    t_load = time.perf_counter() - t0_load
    print(f"Modelo carregado com sucesso em {t_load:.2f}s!")

    if cuda_avail:
        mem_alloc = torch.cuda.memory_allocated(0) / (1024**3)
        mem_res = torch.cuda.memory_reserved(0) / (1024**3)
        print(f"VRAM alocada: {mem_alloc:.2f} GB | VRAM reservada: {mem_res:.2f} GB")

    # 2. Teste 1: Sentença canônica mínima (Aquecimento da GPU)
    print("\n--- [2] Teste Canônico Mínimo (Warmup) ---")
    sample_text = (
        "SENTENÇA. O autor pediu a restituição de valores. "
        "Ante o exposto, JULGO PROCEDENTE o pedido inicial, integralmente, "
        "com resolução do mérito."
    )
    t0_inf = time.perf_counter()
    warmup_res = model.decide(state=sample_text, questions=QUESTIONS, think=False, tta=1)
    t_warmup = time.perf_counter() - t0_inf
    print(f"Latência do aquecimento: {t_warmup:.3f}s ({t_warmup * 1000:.1f} ms)!")
    print(f"Resposta do aquecimento:\n{json.dumps(warmup_res, indent=2, ensure_ascii=False)}")

    # 3. Teste Fixture (12 Casos do CausaGanha)
    print("\n--- [3] Benchmark Completo da Fixture (12 casos) ---")
    fixture_path = "/content/thx02_spike.jsonl"
    if not os.path.exists(fixture_path):
        fixture_path = "tests/fixtures/thx02_spike.jsonl"

    documents = []
    with open(fixture_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                documents.append(json.loads(line))
    print(f"Carregados {len(documents)} documentos da fixture {fixture_path}.")

    results = []
    latencies = []
    correct_outcome = 0
    correct_winner = 0

    for i, doc in enumerate(documents, start=1):
        doc_id = doc["id"]
        text = doc["text"]
        gold = doc.get("gold", {})
        gold_out = gold.get("outcome")
        gold_win = gold.get("winner")

        t0 = time.perf_counter()
        raw_res = model.decide(state=text, questions=QUESTIONS, think=False, tta=1)
        elapsed_s = time.perf_counter() - t0
        elapsed_ms = elapsed_s * 1000.0
        latencies.append(elapsed_ms)

        # Decodificação
        answers = raw_res.get("answers", {})
        raw_out = answers.get("outcome", {}).get("choice", "unknown")
        outcome = OUTCOME_MAP.get(raw_out, "unknown")
        prob_out = answers.get("outcome", {}).get("probability", 0.0)

        raw_rec = answers.get("recorrente_polo", {}).get("choice")
        winner_direct = answers.get("winner_direct", {}).get("choice")
        resolved_win = resolve_winner(outcome, raw_rec)
        winner = winner_direct if winner_direct in {"A", "P", "draw"} else resolved_win

        match_out = outcome == gold_out
        match_win = winner == gold_win
        if match_out:
            correct_outcome += 1
        if match_win:
            correct_winner += 1

        mark_out = "✓" if match_out else "✗"
        mark_win = "✓" if match_win else "✗"
        print(
            f"[{i:02d}/{len(documents):02d}] {doc_id} ({len(text)} chars) -> Latência: {elapsed_ms:.1f} ms | Resultado: {outcome} ({prob_out:.1%}) [{mark_out}] | Vencedor: {winner} [{mark_win}]"
        )

        results.append(
            {
                "id": doc_id,
                "latency_ms": elapsed_ms,
                "outcome": outcome,
                "outcome_probability": prob_out,
                "gold_outcome": gold_out,
                "match_outcome": match_out,
                "winner": winner,
                "gold_winner": gold_win,
                "match_winner": match_win,
                "raw_response": raw_res,
            }
        )

    # Estatísticas
    latencies.sort()
    n = len(latencies)
    p50 = latencies[int(n * 0.50)]
    p95 = latencies[min(int(n * 0.95), n - 1)]
    p99 = latencies[min(int(n * 0.99), n - 1)]
    avg_ms = sum(latencies) / n

    summary = {
        "hardware": device_name,
        "cuda_available": cuda_avail,
        "vram_gb": vram_gb,
        "total_documents": n,
        "outcome_accuracy": correct_outcome / n if n else 0.0,
        "winner_accuracy": correct_winner / n if n else 0.0,
        "latency_p50_ms": p50,
        "latency_p95_ms": p95,
        "latency_p99_ms": p99,
        "latency_avg_ms": avg_ms,
        "throughput_decisions_per_sec": 1000.0 / avg_ms if avg_ms > 0 else 0.0,
        "model_load_time_seconds": t_load,
    }

    print("\n" + "=" * 65)
    print("RESUMO DO BENCHMARK THX-02 NA GPU (COLAB T4)")
    print("=" * 65)
    for k, v in summary.items():
        print(f"{k}: {v}")

    # Salva arquivos de saída
    os.makedirs("/content/output", exist_ok=True)
    with open("/content/output/colab_thx02_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    with open("/content/output/colab_thx02_results.jsonl", "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("\nResultados gravados em /content/output/colab_thx02_summary.json e results.jsonl!")


if __name__ == "__main__":
    main()

THX-02 COLAB GPU BENCHMARK (NVIDIA TESLA T4)


CUDA Available: True
Device: Tesla T4
Total VRAM: 14.56 GB
PyTorch Version: 2.11.0+cu130
Python Version: 3.13.16



--- [1] Carregando modelo THX-02 em GPU (bfloat16) ---


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

... (13 outputs left out)

[09/12] s09 (173 chars) -> Latência: 4673.9 ms | Resultado: provido (0.0%) [✗] | Vencedor: P [✗]


[10/12] s10 (145 chars) -> Latência: 4511.2 ms | Resultado: não provido (0.0%) [✗] | Vencedor: A [✗]


[11/12] s11 (173 chars) -> Latência: 4563.9 ms | Resultado: parcialmente provido (0.0%) [✗] | Vencedor: A [✗]


[12/12] s12 (160 chars) -> Latência: 4585.7 ms | Resultado: não conhecido (0.0%) [✗] | Vencedor: P [✗]

RESUMO DO BENCHMARK THX-02 NA GPU (COLAB T4)
hardware: Tesla T4
cuda_available: True
vram_gb: 14.56317138671875
total_documents: 12
outcome_accuracy: 0.0
winner_accuracy: 0.0
latency_p50_ms: 4487.612420000005
latency_p95_ms: 4673.870695000005
latency_p99_ms: 4673.870695000005
latency_avg_ms: 4474.914768916686
throughput_decisions_per_sec: 0.2234679433329377
model_load_time_seconds: 38.75562032899995

Resultados gravados em /content/output/colab_thx02_summary.json e results.jsonl!


*File Operation*: `download` on `/content/output/colab_thx02_summary.json`

*File Operation*: `download` on `/content/output/colab_thx02_results.jsonl`